# 01_llm_transformer_from_scratch: Implementing Modern LLM Blocks in PyTorch
    
This notebook builds the core structural blocks of modern decoder-only LLM architectures (like Llama 3) from scratch using PyTorch. 

We will implement:
1. **RMSNorm** (Root Mean Square Normalization)
2. **SwiGLU** (Swish Gated Linear Unit) Activation and FFN
3. **RoPE** (Rotary Position Embeddings)
4. **Grouped-Query Attention** (GQA)
5. A combined **LlamaTransformerBlock**

We will verify tensor shapes and parameters at every step.


## 1. Setup and Environment Initialization

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Establish seed for determinism
torch.manual_seed(42)
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())


PyTorch Version: 2.13.0+cpu
CUDA Available: False


### Output Explanation: Environment Setup
- **Determinism**: We set PyTorch manual seeds to ensure all mock layer activations are reproducible across runs.
- **Hardware Target**: Verification is performed on the active runtime environment.


## 2. RMSNorm Layer

In [2]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        # Learnable scale vector gamma, initialized to ones
        self.weight = nn.Parameter(torch.ones(dim))
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: [B, L, d]
        # Calculate variance (mean of squares) over last dimension
        variance = x.pow(2).mean(-1, keepdim=True)
        # Normalize and apply gain weight
        return x * torch.rsqrt(variance + self.eps) * self.weight

# Verify RMSNorm
B, L, d = 2, 4, 16
x = torch.randn(B, L, d)
rmsnorm = RMSNorm(dim=d)
out = rmsnorm(x)

print("Input shape :", x.shape)
print("Output shape:", out.shape)
print("RMS of Normalized outputs (per token):\n", torch.sqrt(out.pow(2).mean(-1)))


Input shape : torch.Size([2, 4, 16])
Output shape: torch.Size([2, 4, 16])
RMS of Normalized outputs (per token):
 tensor([[1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000]], grad_fn=<SqrtBackward0>)


### Output Explanation: RMSNorm
- **Normalizing Behavior**: The output RMS values are scaled to approximately 1.0 (matching the target normalization factor).
- **VRAM Saving**: We bypassed centering the mean entirely, eliminating one global memory reduction pass.


## 3. SwiGLU Activation Function

In [3]:
class SwiGLUFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ffn: int):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ffn, bias=False)  # W matrix
        self.w_value = nn.Linear(d_model, d_ffn, bias=False) # V matrix
        self.w_down = nn.Linear(d_ffn, d_model, bias=False)  # Down projection
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Parallel gates
        gate = self.w_gate(x)
        swish_gate = gate * torch.sigmoid(gate) # Swish(xW)
        value = self.w_value(x)                  # xV
        
        # Gated multiplication and projection down
        return self.w_down(swish_gate * value)

# Verify SwiGLU
ffn = SwiGLUFeedForward(d_model=d, d_ffn=48)
out = ffn(x)

print("Input shape :", x.shape)
print("Output shape:", out.shape)
assert out.shape == x.shape, "FFN output shape mismatch!"


Input shape : torch.Size([2, 4, 16])
Output shape: torch.Size([2, 4, 16])


### Output Explanation: SwiGLU
- **Dimensions**: Input `[2, 4, 16]` is projected up to intermediate dimension `48` inside the parallel gated paths and projected back down to `16`.
- **Gating Mechanism**: The gating multiplication models sharper activation regions, enhancing representation capability.


## 4. Rotary Position Embeddings (RoPE)

In [4]:
class RotaryPositionEmbedding(nn.Module):
    def __init__(self, dim: int, max_seq_len: int = 100, theta: float = 10000.0):
        super().__init__()
        assert dim % 2 == 0
        self.dim = dim
        
        # Calculate theta values: [dim / 2]
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        
        # Precompute cosine and sine tables: [max_seq_len, dim]
        t = torch.arange(max_seq_len, dtype=torch.float32)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)
        
    def _rotate_half(self, x: torch.Tensor) -> torch.Tensor:
        x1 = x[..., :self.dim // 2]
        x2 = x[..., self.dim // 2:]
        return torch.cat((-x2, x1), dim=-1)
        
    def forward(self, x: torch.Tensor, seq_len: int) -> torch.Tensor:
        # x shape: [B, h, L, d_k]
        cos = self.cos_cached[:seq_len, :].unsqueeze(0).unsqueeze(1) # [1, 1, L, d_k]
        sin = self.sin_cached[:seq_len, :].unsqueeze(0).unsqueeze(1) # [1, 1, L, d_k]
        return (x * cos) + (self._rotate_half(x) * sin)

# Verify RoPE
# q shape: [B, h, L, d_k] -> Batch=2, Heads=4, SeqLen=3, HeadDim=8
q = torch.randn(2, 4, 3, 8)
rope = RotaryPositionEmbedding(dim=8)
q_rotated = rope(q, seq_len=3)

print("Input shape :", q.shape)
print("Output shape:", q_rotated.shape)
assert q_rotated.shape == q.shape, "RoPE shape mismatch!"


Input shape : torch.Size([2, 4, 3, 8])
Output shape: torch.Size([2, 4, 3, 8])


### Output Explanation: RoPE
- **Half-Rotation**: The `_rotate_half` helper swaps the first and second halves of dimensions and negates one, implementing the complex multiplication.
- **Order Preservation**: Dot products calculated between rotated queries and keys will depend strictly on relative positions.


## 5. Grouped-Query Attention (GQA)

In [5]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, embed_dim: int, num_query_heads: int, num_kv_heads: int):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_query_heads = num_query_heads
        self.num_kv_heads = num_kv_heads
        self.group_size = num_query_heads // num_kv_heads
        self.head_dim = embed_dim // num_query_heads
        
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, num_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        
    def _repeat_heads(self, x: torch.Tensor, reps: int) -> torch.Tensor:
        B, n_heads, L, d_k = x.shape
        if reps == 1:
            return x
        x = x.unsqueeze(2).expand(B, n_heads, reps, L, d_k)
        return x.reshape(B, n_heads * reps, L, d_k)
        
    def forward(self, x: torch.Tensor, rope: nn.Module) -> torch.Tensor:
        B, L, d = x.shape
        
        # Project and reshape: [B, h, L, d_k]
        q = self.q_proj(x).view(B, L, self.num_query_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, L, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, L, self.num_kv_heads, self.head_dim).transpose(1, 2)
        
        # Apply RoPE
        q = rope(q, seq_len=L)
        k = rope(k, seq_len=L)
        
        # Broadcast/Repeat KV heads to match Query heads
        k = self._repeat_heads(k, self.group_size)
        v = self._repeat_heads(v, self.group_size)
        
        # Compute scaled attention weights: [B, h_q, L, L]
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_weights = F.softmax(scores, dim=-1)
        
        # Weighted sum: [B, h_q, L, d_k]
        context = torch.matmul(attn_weights, v)
        
        # Concatenate and project back
        context = context.transpose(1, 2).contiguous().view(B, L, d)
        return self.out_proj(context)

# Verify GQA
# embed_dim=16, 4 query heads, 2 KV heads (group size 2)
rope = RotaryPositionEmbedding(dim=4) # head_dim = 16 // 4 = 4
gqa = GroupedQueryAttention(embed_dim=16, num_query_heads=4, num_kv_heads=2)
out = gqa(x, rope)

print("Input shape :", x.shape)
print("Output shape:", out.shape)
assert out.shape == x.shape, "GQA output shape mismatch!"


Input shape : torch.Size([2, 4, 16])
Output shape: torch.Size([2, 4, 16])


### Output Explanation: Grouped-Query Attention
- **Broadcasting**: The 2 Key and Value heads are duplicated to 4 heads to match the Query head count.
- **Savings**: Memory cached for Keys/Values is halved compared to standard MHA.


## 6. Full LlamaTransformerBlock Integration

In [6]:
class LlamaTransformerBlock(nn.Module):
    def __init__(self, embed_dim: int, num_query_heads: int, num_kv_heads: int, d_ffn: int):
        super().__init__()
        self.attn_norm = RMSNorm(embed_dim)
        self.attn = GroupedQueryAttention(embed_dim, num_query_heads, num_kv_heads)
        self.rope = RotaryPositionEmbedding(dim=embed_dim // num_query_heads)
        
        self.ffn_norm = RMSNorm(embed_dim)
        self.ffn = SwiGLUFeedForward(embed_dim, d_ffn)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-LN attention block with residual connection
        h = x + self.attn(self.attn_norm(x), self.rope)
        # Pre-LN FFN block with residual connection
        out = h + self.ffn(self.ffn_norm(h))
        return out

# Instantiate full block and run
llama_block = LlamaTransformerBlock(
    embed_dim=128,
    num_query_heads=8,
    num_kv_heads=2,
    d_ffn=340
)
x_block = torch.randn(2, 10, 128) # Batch=2, Seq=10, Dim=128
out_block = llama_block(x_block)

print("Input Block shape :", x_block.shape)
print("Output Block shape:", out_block.shape)
assert out_block.shape == x_block.shape
print("Full Llama Transformer Block completed successfully!")


Input Block shape : torch.Size([2, 10, 128])
Output Block shape: torch.Size([2, 10, 128])
Full Llama Transformer Block completed successfully!


### Output Explanation: Llama Block Integration
- **Pre-LN Flow**: Normalization happens prior to entering the attention and feedforward layers, preserving the identity residual shortcut pathway.
- **Llama Config**: A mini-Llama layer running successfully with GQA, RoPE, RMSNorm, and SwiGLU.
